# Model Monitoring with Evidently AI

This notebook loads batch prediction results, compares them to the test reference data, and calculates model validation and data drift metrics using Evidently AI.

In [16]:
import json
import os
import subprocess
import sys
from pathlib import Path

try:
    import pandas as pd
    import joblib
    from evidently.legacy.report.report import Report
    from evidently.legacy.metric_preset import DataDriftPreset, ClassificationPreset
    from evidently.legacy.pipeline.column_mapping import ColumnMapping
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'evidently'])
    import pandas as pd
    import joblib
    from evidently.legacy.report.report import Report
    from evidently.legacy.metric_preset import DataDriftPreset, ClassificationPreset
    from evidently.legacy.pipeline.column_mapping import ColumnMapping

In [17]:
# Paths and configuration
BASE_DIR = Path('/home/azureuser/cloudfiles/code/Users/Mahesh113274/MLOps_Azure')
BATCH_RESULTS_DIR = BASE_DIR / 'outputs' / 'batch_results'
BATCH_RESULTS_FILE = BATCH_RESULTS_DIR / 'batch_predictions.csv'
REFERENCE_X_TEST = BASE_DIR / 'outputs' / 'x_test.pkl'
REFERENCE_Y_TEST = BASE_DIR / 'outputs' / 'y_test.pkl'
MONITORING_OUTPUT_DIR = BATCH_RESULTS_DIR / 'model_monitoring'
MONITORING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Batch results file:', BATCH_RESULTS_FILE)
print('Reference x_test file:', REFERENCE_X_TEST)
print('Reference y_test file:', REFERENCE_Y_TEST)
print('Monitoring output dir:', MONITORING_OUTPUT_DIR)

Batch results file: /home/azureuser/cloudfiles/code/Users/Mahesh113274/MLOps_Azure/outputs/batch_results/batch_predictions.csv
Reference x_test file: /home/azureuser/cloudfiles/code/Users/Mahesh113274/MLOps_Azure/outputs/x_test.pkl
Reference y_test file: /home/azureuser/cloudfiles/code/Users/Mahesh113274/MLOps_Azure/outputs/y_test.pkl
Monitoring output dir: /home/azureuser/cloudfiles/code/Users/Mahesh113274/MLOps_Azure/outputs/batch_results/model_monitoring


In [18]:
# Load batch prediction results
if not BATCH_RESULTS_FILE.exists():
    raise FileNotFoundError(f'Batch results not found: {BATCH_RESULTS_FILE}')

batch_df = pd.read_csv(BATCH_RESULTS_FILE)
print('Loaded batch prediction results with shape:', batch_df.shape)
print(batch_df.columns.tolist())

Loaded batch prediction results with shape: (134807, 33)
['ID', 'Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'prediction', 'fraud_probability']


In [19]:
# Load reference test data for model validation
if not REFERENCE_X_TEST.exists() or not REFERENCE_Y_TEST.exists():
    raise FileNotFoundError('Reference test data not found in outputs directory')

x_test = joblib.load(REFERENCE_X_TEST)
y_test = joblib.load(REFERENCE_Y_TEST)
reference_df = x_test.copy()
reference_df['Class'] = y_test
print('Loaded reference test data with shape:', reference_df.shape)

Loaded reference test data with shape: (30000, 32)


In [20]:
# Prepare current dataset for analysis
current_df = batch_df.copy()
if 'Class' in current_df.columns:
    print('Target column Class found in batch results.')
else:
    print('Target column Class not found in batch results; drift-only monitoring will be performed.')

if 'prediction' not in current_df.columns:
    raise KeyError('Expected column prediction is missing from batch results')

print('Current dataset columns:', current_df.columns.tolist())

Target column Class not found in batch results; drift-only monitoring will be performed.
Current dataset columns: ['ID', 'Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'prediction', 'fraud_probability']


In [24]:
# Define column mapping for Evidently - focus on feature drift detection
# Batch predictions don't have ground truth, so we do drift analysis only
feature_columns = [col for col in current_df.columns if col not in ['Class', 'prediction', 'fraud_probability']]
column_mapping = ColumnMapping(
    prediction=None,  # Skip prediction since reference doesn't have it
    numerical_features=feature_columns,
    target=None,  # No target for drift analysis (batch has no ground truth)
    task=None
)
print('Column mapping created for feature drift analysis')
print('Number of features:', len(feature_columns))


Column mapping created for feature drift analysis
Number of features: 31


In [25]:
# Run data drift monitoring
drift_report = Report(metrics=[DataDriftPreset()])
drift_report.run(reference_data=reference_df, current_data=current_df, column_mapping=column_mapping)
drift_html = MONITORING_OUTPUT_DIR / 'data_drift_report.html'
drift_report.save_html(str(drift_html))
print('Saved data drift report to:', drift_html)

Saved data drift report to: /home/azureuser/cloudfiles/code/Users/Mahesh113274/MLOps_Azure/outputs/batch_results/model_monitoring/data_drift_report.html


In [26]:
# Note: Classification performance monitoring is not available for batch predictions
# because batch data does not contain ground truth labels (Class column)
# Only feature drift analysis is performed in this notebook
print('Batch prediction results do not contain ground truth labels.')
print('Skipping classification performance monitoring.')
print('Feature drift analysis completed successfully.')
print('Check data_drift_report.html for detailed drift metrics.')


Batch prediction results do not contain ground truth labels.
Skipping classification performance monitoring.
Feature drift analysis completed successfully.
Check data_drift_report.html for detailed drift metrics.


In [27]:
# Save a summary of monitoring results
summary = {
    'reference_shape': reference_df.shape,
    'current_shape': current_df.shape,
    'batch_results_file': str(BATCH_RESULTS_FILE),
    'monitoring_output_dir': str(MONITORING_OUTPUT_DIR),
    'has_ground_truth': 'Class' in current_df.columns,
}
summary_file = MONITORING_OUTPUT_DIR / 'monitoring_summary.json'
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved monitoring summary to:', summary_file)
print('Model monitoring completed successfully.')

Saved monitoring summary to: /home/azureuser/cloudfiles/code/Users/Mahesh113274/MLOps_Azure/outputs/batch_results/model_monitoring/monitoring_summary.json
Model monitoring completed successfully.
